In [61]:
import pandas as pd
import numpy as np
import math
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

In [62]:
raw_data = pd.read_csv("BOM.csv").set_axis(list(range(23)), axis=1)
raw_data = raw_data.drop(0, axis=1)

In [63]:
Y = raw_data[22]
X = raw_data.drop(columns=[22])

In [64]:
dist_attr = [1,7,9,10,21]
cont_attr = [x for x in range(1,23) if x not in dist_attr and x != 22]

binary_data = [21]

In [65]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)

In [66]:
raw_data.isna().sum()

1         0
2      1485
3      1261
4      3261
5     62790
6     69835
7     10326
8     10263
9     10566
10     4228
11     1767
12     3062
13     2654
14     4507
15    15065
16    15028
17    55888
18    59358
19     1767
20     3609
21     3261
22     3267
dtype: int64

In [67]:
for col in X.columns:
    if col in cont_attr:
        mean_val = x_train[col].mean()
        x_train[col] = x_train[col].fillna(mean_val)
        x_test[col] = x_test[col].fillna(mean_val)
    else:
        mode_val = x_train[col].mode()[0]
        x_train[col] = x_train[col].fillna(mode_val)
        x_test[col] = x_test[col].fillna(mode_val)

In [68]:
scaler = MinMaxScaler()
x_train[cont_attr] = scaler.fit_transform(x_train[cont_attr])
x_test[cont_attr] = scaler.transform(x_test[cont_attr])

In [69]:
ord_enc = OrdinalEncoder()
x_train[binary_data] = ord_enc.fit_transform(x_train[binary_data])
x_test[binary_data] = ord_enc.transform(x_test[binary_data])


In [70]:
cat_data = [x for x in dist_attr if x not in binary_data]


In [71]:
x_train_cat = pd.get_dummies(x_train[cat_data])
x_test_cat = pd.get_dummies(x_test[cat_data])
x_train_cat, x_test_cat = x_train_cat.align(x_test_cat, join='left', axis=1, fill_value=0)

In [72]:
x_train = x_train.drop(columns=cat_data).join(x_train_cat)
x_test = x_test.drop(columns=cat_data).join(x_test_cat)

x_train.columns = x_train.columns.astype(str)
x_test.columns = x_test.columns.astype(str)


In [73]:
best_k = 0
best_f1 = 0
print(y_train.isna().sum().max())

for k in range(1, 51):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(x_train, y_train)

    y_pred = knn.predict(x_test)
    f1 = f1_score(y_test, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_k = k

print("Best K:", best_k)
print("Best F1:", best_f1)

2565


ValueError: Input contains NaN